# BERTScore F1

Plain BERTScore F1 for the predictions logged by existing MLflow runs -- no
judge, no rewriting, just the metric.

## Config

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
REPO_ROOT = "/home/phuc/code-sum"

RUN_IDS = {
    "few_shot_llm": "8ed48f980c5540e8b029056b268867e8",
    "few_shot_all_context": "7e938318a5974bcaaf0c424d43bae43e",
    "zero_shot": "5987585716d54933b79acdbedaaaad15",
    "metagente": "512522dd8ba14ff4b8930906cea34a0f",
}

# Same model as the project's own bertscore_f1 metric (README: "average BERTScore F1 (roberta-large)")
BERTSCORE_MODEL_TYPE = "roberta-large"
BERTSCORE_LANG = "en"
BERTSCORE_BATCH_SIZE = 32
BERTSCORE_DEVICE = "cuda"  # falls back to "cpu" below if unavailable

In [ ]:
import pathlib

import mlflow
import pandas as pd
import torch
from mlflow import MlflowClient

if not torch.cuda.is_available():
    BERTSCORE_DEVICE = "cpu"

mlflow.set_tracking_uri("http://127.0.0.1:5000")
client = MlflowClient()

## Load predictions from MLflow

Each run logged a `predictions/*.csv` artifact with columns `id, project,
func_name, run, reference, prediction`.

In [ ]:
def load_predictions(run_name: str, run_id: str) -> pd.DataFrame:
    local_dir = client.download_artifacts(run_id, "predictions")
    csvs = list(pathlib.Path(local_dir).glob("*.csv"))
    assert len(csvs) == 1, f"expected exactly one predictions csv, found {csvs}"
    df = pd.read_csv(csvs[0])
    df["system"] = run_name
    return df


pred_df = pd.concat(
    [load_predictions(name, run_id) for name, run_id in RUN_IDS.items()],
    ignore_index=True,
)
pred_df["system"].value_counts()

## BERTScore F1

Scored in a single batched call via the `bert-score` package (already a
project dependency).

In [ ]:
from bert_score import BERTScorer

scorer = BERTScorer(
    model_type=BERTSCORE_MODEL_TYPE,
    lang=BERTSCORE_LANG,
    rescale_with_baseline=False,
    device=BERTSCORE_DEVICE,
)

_, _, f1 = scorer.score(
    pred_df["prediction"].tolist(), pred_df["reference"].tolist(), batch_size=BERTSCORE_BATCH_SIZE
)
pred_df["bertscore_f1"] = f1.tolist()
pred_df[["system", "id", "reference", "prediction", "bertscore_f1"]].head()

## Aggregate per system

In [ ]:
pred_df.groupby("system")["bertscore_f1"].agg(["mean", "std", "count"])